In [ ]:
import os
import json
import shutil
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from PIL import Image

print('TensorFlow version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs detected: {len(gpus)}')
if gpus:
    for gpu in gpus:
        print(' -', gpu)
    # Allow memory growth — prevents OOM on Kaggle
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print('No GPU found. Go to Settings → Accelerator → GPU T4 x2')

KAGGLE_INPUT   = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')

base = Path('/kaggle/input/datasets/abdallahalidev/plantvillage-dataset')

DATA_DIR = None
for root, dirs, files in os.walk(base):
    root_path = Path(root)
    images = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if images:
        candidate = root_path.parent
        subdirs = [d for d in candidate.iterdir() if d.is_dir()]
        if len(subdirs) >= 5:
            DATA_DIR = candidate
            break

assert DATA_DIR is not None, 'Could not locate class folders inside the dataset'
print(f'DATA_DIR : {DATA_DIR}')
print(f'Classes  : {len(list(DATA_DIR.iterdir()))}')
print(f'Sample   : {[d.name for d in list(DATA_DIR.iterdir())[:4]]}')

IMG_SIZE        = 224     
BATCH_SIZE      = 32      
HEAD_EPOCHS     = 15      
FINETUNE_EPOCHS = 25      
INITIAL_LR      = 1e-3    
FINETUNE_LR     = 1e-5    
UNFREEZE_LAYERS = 30      
VAL_SPLIT       = 0.15
TEST_SPLIT      = 0.10
RANDOM_SEED     = 42

MODEL_DIR       = KAGGLE_WORKING / 'plant_disease_model'
SPLIT_DIR       = KAGGLE_WORKING / 'dataset_split'
CHECKPOINT_DIR  = MODEL_DIR / 'checkpoints'
LOG_DIR         = MODEL_DIR / 'logs'

for d in [MODEL_DIR, SPLIT_DIR, CHECKPOINT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Configuration:')
print(f'  IMG_SIZE        = {IMG_SIZE}')
print(f'  BATCH_SIZE      = {BATCH_SIZE}')
print(f'  HEAD_EPOCHS     = {HEAD_EPOCHS}')
print(f'  FINETUNE_EPOCHS = {FINETUNE_EPOCHS}')
print(f'  Output dir      = {MODEL_DIR}')

if (SPLIT_DIR / 'train').exists() and any((SPLIT_DIR / 'train').iterdir()):
    print('Split already exists — skipping copy step')
else:
    print('Splitting dataset (this copies files — takes 2–5 min)...')
    total_copied = 0

    for class_dir in sorted(DATA_DIR.iterdir()):
        if not class_dir.is_dir():
            continue

        images = (
            list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.JPG')) +
            list(class_dir.glob('*.png')) + list(class_dir.glob('*.PNG')) +
            list(class_dir.glob('*.jpeg'))
        )
        if not images:
            continue

        train_imgs, temp = train_test_split(
            images, test_size=(VAL_SPLIT + TEST_SPLIT), random_state=RANDOM_SEED
        )
        val_ratio = VAL_SPLIT / (VAL_SPLIT + TEST_SPLIT)
        val_imgs, test_imgs = train_test_split(
            temp, test_size=(1 - val_ratio), random_state=RANDOM_SEED
        )

        for split, imgs in [('train', train_imgs), ('val', val_imgs), ('test', test_imgs)]:
            dest = SPLIT_DIR / split / class_dir.name
            dest.mkdir(parents=True, exist_ok=True)
            for img_path in imgs:
                shutil.copy2(img_path, dest / img_path.name)
                total_copied += 1

    print(f'Done! {total_copied} images copied to {SPLIT_DIR}')

for split in ['train', 'val', 'test']:
    count = sum(1 for _ in (SPLIT_DIR / split).rglob('*.jpg'))
    count += sum(1 for _ in (SPLIT_DIR / split).rglob('*.JPG'))
    count += sum(1 for _ in (SPLIT_DIR / split).rglob('*.png'))
    print(f'  {split:<6}: {count:>6} images')

train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest',
)

eval_datagen = ImageDataGenerator(rescale=1.0 / 255)

flow_args = dict(
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    seed=RANDOM_SEED,
)

train_gen = train_datagen.flow_from_directory(
    str(SPLIT_DIR / 'train'), shuffle=True, **flow_args
)
val_gen = eval_datagen.flow_from_directory(
    str(SPLIT_DIR / 'val'), shuffle=False, **flow_args
)
test_gen = eval_datagen.flow_from_directory(
    str(SPLIT_DIR / 'test'), shuffle=False, **flow_args
)

NUM_CLASSES  = train_gen.num_classes
CLASS_NAMES  = list(train_gen.class_indices.keys())

class_map = {str(v): k for k, v in train_gen.class_indices.items()}
with open(MODEL_DIR / 'class_indices.json', 'w') as f:
    json.dump(class_map, f, indent=2)

print(f'Classes : {NUM_CLASSES}')
print(f'Train   : {train_gen.n} images')
print(f'Val     : {val_gen.n} images')
print(f'Test    : {test_gen.n} images')
print(f'class_indices.json saved')

images, labels = next(train_gen)

fig, axes = plt.subplots(3, 4, figsize=(15, 10))
fig.suptitle('Training samples (with augmentation)', fontsize=14, fontweight='bold')
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i])
    idx = np.argmax(labels[i])
    name = CLASS_NAMES[idx].replace('___', '\n').replace('_', ' ')
    ax.set_title(name, fontsize=7)
    ax.axis('off')
plt.tight_layout()
plt.savefig(str(MODEL_DIR / 'sample_images.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Saved sample_images.png')

counts = Counter(train_gen.classes)
sorted_items = sorted(counts.items(), key=lambda x: x[1])
labels_sorted = [CLASS_NAMES[i].replace('___', ' — ').replace('_', ' ') for i, _ in sorted_items]
values_sorted = [c for _, c in sorted_items]

fig, ax = plt.subplots(figsize=(16, 6))
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(values_sorted)))
ax.barh(labels_sorted, values_sorted, color=colors)
ax.set_title('Training images per class (sorted by count)', fontsize=13, fontweight='bold')
ax.set_xlabel('Image count')
ax.tick_params(axis='y', labelsize=7)
plt.tight_layout()
plt.savefig(str(MODEL_DIR / 'class_distribution.png'), dpi=120, bbox_inches='tight')
plt.show()

y_train = train_gen.classes
unique_classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight='balanced',
    classes=unique_classes,
    y=y_train,
)
CLASS_WEIGHTS = dict(zip(unique_classes.tolist(), weights.tolist()))

sorted_w = sorted(CLASS_WEIGHTS.items(), key=lambda x: x[1])
print('Most over-represented (low weight):')
for idx, w in sorted_w[:3]:
    print(f'  {CLASS_NAMES[idx]:<55} {w:.3f}')
print('Most under-represented (high weight):')
for idx, w in sorted_w[-3:]:
    print(f'  {CLASS_NAMES[idx]:<55} {w:.3f}')

def build_model(num_classes: int) -> keras.Model:
    """EfficientNetB3 + custom classification head."""
    base_model = EfficientNetB3(
        weights='imagenet',  
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
    )
    base_model.trainable = False  

    inputs  = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x       = base_model(inputs, training=False)  
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dropout(0.4)(x)
    x       = layers.Dense(512, activation='relu')(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=INITIAL_LR),
        loss='categorical_crossentropy',
        metrics=[
            'accuracy',
            keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc'),
        ],
    )
    return model

model = build_model(NUM_CLASSES)  
print("Model built successfully!")
model.summary()

def get_callbacks(phase: str) -> list:
    return [
        callbacks.ModelCheckpoint(
            filepath=str(CHECKPOINT_DIR / f'best_{phase}.keras'),
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1,
        ),
        callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=8,
            restore_best_weights=True,
            verbose=1,
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.3,
            patience=4,
            min_lr=1e-7,
            verbose=1,
        ),
        callbacks.CSVLogger(
            filename=str(LOG_DIR / f'{phase}_log.csv'),
            append=True,
        ),
    ]

print('Callbacks ready')

print('=' * 55)
print('  Phase 1 — Head training (base frozen)')
print('=' * 55)

history_phase1 = model.fit(
    train_gen,
    epochs=HEAD_EPOCHS,
    validation_data=val_gen,
    class_weight=CLASS_WEIGHTS,
    callbacks=get_callbacks('phase1'),
    verbose=1,
)

best_val = max(history_phase1.history['val_accuracy'])
print(f'\nPhase 1 best val_accuracy: {best_val*100:.2f}%')

base_model = model.layers[1]   
base_model.trainable = True

for layer in base_model.layers[:-UNFREEZE_LAYERS]:
    layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(f'Unfrozen {trainable_count} layers in EfficientNetB3')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=FINETUNE_LR),
    loss='categorical_crossentropy',
    metrics=[
        'accuracy',
        keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc'),
    ],
)

print('=' * 55)
print('  Phase 2 — Fine-tuning top layers')
print('=' * 55)

history_phase2 = model.fit(
    train_gen,
    epochs=FINETUNE_EPOCHS,
    validation_data=val_gen,
    class_weight=CLASS_WEIGHTS,
    callbacks=get_callbacks('phase2'),
    verbose=1,
)

best_val2 = max(history_phase2.history['val_accuracy'])
print(f'\nPhase 2 best val_accuracy: {best_val2*100:.2f}%')


def plot_history(h1, h2):
    def merge(key):
        return h1.history.get(key, []) + h2.history.get(key, [])

    acc      = merge('accuracy')
    val_acc  = merge('val_accuracy')
    loss     = merge('loss')
    val_loss = merge('val_loss')
    epochs   = range(1, len(acc) + 1)
    boundary = len(h1.history['accuracy'])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Training History — Plant Disease Detection', fontsize=14, fontweight='bold')

    axes[0].plot(epochs, acc,     label='Train',      color='#2ecc71', lw=2)
    axes[0].plot(epochs, val_acc, label='Validation', color='#3498db', lw=2, ls='--')
    axes[0].axvline(boundary, color='#e74c3c', ls=':', lw=1.5, label='Fine-tune start')
    axes[0].set_title('Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(epochs, loss,     label='Train',      color='#e67e22', lw=2)
    axes[1].plot(epochs, val_loss, label='Validation', color='#9b59b6', lw=2, ls='--')
    axes[1].axvline(boundary, color='#e74c3c', ls=':', lw=1.5, label='Fine-tune start')
    axes[1].set_title('Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(str(MODEL_DIR / 'training_history.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved training_history.png')

plot_history(history_phase1, history_phase2)

print('Running test set evaluation...')
test_gen.reset()

predictions = model.predict(test_gen, verbose=1)
y_pred = np.argmax(predictions, axis=1)
y_true = test_gen.classes

top1 = np.mean(y_pred == y_true)
top3_preds = np.argsort(predictions, axis=1)[:, -3:]
top3 = np.mean([y_true[i] in top3_preds[i] for i in range(len(y_true))])

print(f'\n Test Top-1 Accuracy: {top1*100:.2f}%')
print(f' Test Top-3 Accuracy: {top3*100:.2f}%')

report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3)
print('\nClassification Report:')
print(report)

with open(MODEL_DIR / 'classification_report.txt', 'w') as f:
    f.write(f'Top-1 Accuracy: {top1*100:.2f}%\n')
    f.write(f'Top-3 Accuracy: {top3*100:.2f}%\n\n')
    f.write(report)
print('\nSaved classification_report.txt')

cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

n = NUM_CLASSES
fig, ax = plt.subplots(figsize=(max(16, n * 0.5), max(14, n * 0.45)))
sns.heatmap(
    cm_norm, annot=False, cmap='Blues', fmt='.2f',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax,
    linewidths=0.3, vmin=0, vmax=1,
)
ax.set_title('Normalized Confusion Matrix', fontsize=14)
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
plt.xticks(rotation=90, ha='right', fontsize=6)
plt.yticks(rotation=0, fontsize=6)
plt.tight_layout()
plt.savefig(str(MODEL_DIR / 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved confusion_matrix.png')

keras_path = str(MODEL_DIR / 'plant_disease_model_final.keras')
saved_model_path = str(MODEL_DIR / 'saved_model')

model.save(keras_path)
model.save(saved_model_path)

print(f'\n Keras model saved  → {keras_path}')
print(f' SavedModel saved   → {saved_model_path}')
print(f' class_indices.json → {MODEL_DIR / "class_indices.json"}')

print('\nFiles in output directory:')
for f in sorted(MODEL_DIR.rglob('*')):
    if f.is_file():
        size_mb = f.stat().st_size / 1e6
        print(f'  {str(f.relative_to(MODEL_DIR)):<55} {size_mb:.1f} MB')

def parse_class_label(label: str):
    """Parse 'Tomato___Early_blight' → (plant, is_healthy, disease)"""
    parts = label.split('___')
    plant = parts[0].replace('_', ' ').split('(')[0].strip().title()
    if len(parts) < 2 or parts[1].lower() == 'healthy':
        return plant, True, 'Healthy'
    disease = parts[1].replace('_', ' ').strip().title()
    return plant, False, disease

def predict_image(img_path: str) -> dict:
    img = Image.open(img_path).convert('RGB')
    img_resized = img.resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS)
    arr = np.array(img_resized, dtype=np.float32) / 255.0
    arr = np.expand_dims(arr, axis=0)

    probs = model.predict(arr, verbose=0)[0]
    top3_idx = np.argsort(probs)[::-1][:3]

    best_label = CLASS_NAMES[top3_idx[0]]
    plant, is_healthy, disease = parse_class_label(best_label)

    return {
        'plant_name':   plant,
        'is_healthy':   is_healthy,
        'disease_name': disease,
        'confidence':   float(round(probs[top3_idx[0]], 4)),
        'top3': [
            {'label': CLASS_NAMES[i], 'confidence': float(round(probs[i], 4))}
            for i in top3_idx
        ]
    }

test_images = list((SPLIT_DIR / 'test').rglob('*.jpg')) + \
              list((SPLIT_DIR / 'test').rglob('*.JPG')) + \
              list((SPLIT_DIR / 'test').rglob('*.png'))

import random
random.seed(123)
sample_img_path = random.choice(test_images)
result = predict_image(str(sample_img_path))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
img_display = Image.open(sample_img_path)
axes[0].imshow(img_display)
axes[0].set_title('Input image', fontsize=11)
axes[0].axis('off')

labels_bar = [p['label'].replace('___', '\n').replace('_', ' ') for p in result['top3']]
confs_bar  = [p['confidence'] * 100 for p in result['top3']]
colors_bar = ['#2ecc71' if result['is_healthy'] else '#e74c3c'] + ['#95a5a6'] * 2
axes[1].barh(labels_bar[::-1], confs_bar[::-1], color=colors_bar[::-1])
axes[1].set_xlim(0, 100)
axes[1].set_xlabel('Confidence (%)')
axes[1].set_title('Top-3 Predictions', fontsize=11)

fig.suptitle(
    f"Plant: {result['plant_name']}  |  "
    f"{' Healthy' if result['is_healthy'] else '' + result['disease_name']}  |  "
    f"Confidence: {result['confidence']*100:.1f}%",
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig(str(MODEL_DIR / 'inference_test.png'), dpi=120, bbox_inches='tight')
plt.show()

print('\nPrediction result:')
print(json.dumps(result, indent=2))
